## 12.1. Tokenization


In [1]:
!pip install transformers datasets torch


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\GVC\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
from transformers import AutoTokenizer

# 1. Khởi tạo tokenizer của mô hình pretrained (BERT)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Tokenize một đoạn văn bản mẫu
text = ["This movie is awesome!", "I hated the plot."]
tokens = tokenizer(
    text,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# 3. Kiểm tra các keys trong dictionary trả về
print("Các keys của tokens:", tokens.keys())

c:\Users\GVC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Các keys của tokens: KeysView({'input_ids': tensor([[  101,  2023,  3185,  2003, 12476,   999,   102],
        [  101,  1045,  6283,  1996,  5436,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1]])})


## 14. Experiment 1 — Pretrained vs From Scratch

In [3]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score

# 1. Chuẩn bị dữ liệu IMDb (lấy mẫu nhỏ 1000 train / 500 test để chạy nhanh lấy chỉ số)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

dataset = load_dataset("imdb")
train_data = dataset["train"].shuffle(seed=42).select(range(1000))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize_fn, batched=True)
test_data = test_data.map(tokenize_fn, batched=True)

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

# ==========================================
# MODEL A: Transformer From Scratch
# ==========================================
class SmallTransformerScratch(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=128, num_heads=4, hidden_dim=256, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(embed_dim, num_classes)
        
    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x = self.transformer(x)
        x = x.mean(dim=1) # Global average pooling
        return self.fc(x)

# ==========================================
# MODEL B: Pretrained BERT (Fine-tuning)
# ==========================================
class BertFineTune(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :] # Lấy [CLS]
        return self.fc(cls_rep)

# ==========================================
# Hàm Train & Evaluate
# ==========================================
def train_and_eval(model, name, epochs=2):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5 if "BERT" in name else 1e-3)
    criterion = nn.CrossEntropyLoss()
    
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n--- Bắt đầu huấn luyện {name} (Tổng tham số: {num_params:,}) ---")
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
    train_time = time.time() - start_time
    
    # Evaluate
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            logits = model(input_ids, attention_mask)
            predictions = torch.argmax(logits, dim=-1)
            
            preds.extend(predictions.cpu().tolist())
            targets.extend(labels.cpu().tolist())
            
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average="macro")
    
    print(f"--> Kết quả {name}: Accuracy = {acc:.4f} | F1 = {f1:.4f} | Thời gian: {train_time:.2f}s")
    return acc, f1, f1_score, train_time, num_params

# Thống kê kết quả
results = {}
results["From Scratch"] = train_and_eval(SmallTransformerScratch(), "Transformer From Scratch", epochs=3)
results["Pretrained BERT"] = train_and_eval(BertFineTune(), "Pretrained BERT", epochs=2)

Đang sử dụng thiết bị: cpu


HfUriError: Invalid HF URI 'hf://datasets/imdb@e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml'. Repository id must be 'namespace/name', got 'imdb'.